In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ---------- 数据准备 ----------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST('./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST('./data', train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256)

In [2]:
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [4]:
model = MNISTNet().to(device)

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [6]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        optimizer.zero_grad()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    acc = correct / total
    print(f"Epoch [{epoch + 1}/{num_epochs}]  Loss: {avg_loss:.4f}  Train Acc: {acc:.2%}")

Epoch [1/10]  Loss: 0.2282  Train Acc: 93.19%
Epoch [2/10]  Loss: 0.0916  Train Acc: 97.21%
Epoch [3/10]  Loss: 0.0648  Train Acc: 97.96%
Epoch [4/10]  Loss: 0.0492  Train Acc: 98.39%
Epoch [5/10]  Loss: 0.0412  Train Acc: 98.70%
Epoch [6/10]  Loss: 0.0337  Train Acc: 98.84%
Epoch [7/10]  Loss: 0.0281  Train Acc: 99.05%
Epoch [8/10]  Loss: 0.0236  Train Acc: 99.17%
Epoch [9/10]  Loss: 0.0209  Train Acc: 99.31%
Epoch [10/10]  Loss: 0.0187  Train Acc: 99.40%
